In [7]:
# 키워드를 룰맵 기반 매핑 및 영어는 소문자로 통일
import json
import os

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
input_json_path = '../SSU_Datathon2025_공학분야_62199_Final.json'
rule_map_path = 'keyword_rule_map.json'
output_json_path = 'SSU_Datathon2025_공학분야_62199_Real_Final.json'

def load_rule_map(path):
    """
    룰맵 로드: 사용자의 요청에 따라 키(Key)를 정규화하지 않고 원본 그대로 사용
    """
    print("📂 룰맵 로딩 중...")
    rule_map = {}
    
    if not os.path.exists(path):
        print(f"⚠️ 경고: 룰맵 파일({path})이 없습니다. 변환 없이 진행됩니다.")
        return {}

    try:
        with open(path, 'r', encoding='utf-8') as f:
            rule_data = json.load(f)

        # 룰맵 구조 처리
        items = []
        if isinstance(rule_data, dict):
            if "exact_map" in rule_data:
                # [변경점] replace(" ", "").lower() 제거 -> 원본 키 사용
                for v, c in rule_data["exact_map"].items():
                    rule_map[v.strip()] = c
            if "variants_list" in rule_data:
                items = rule_data["variants_list"]
        elif isinstance(rule_data, list):
            items = rule_data

        for item in items:
            if "variant_norm" in item and "chosen" in item:
                # [변경점] variant_norm도 원본 그대로 키로 사용
                key = str(item["variant_norm"]).strip()
                rule_map[key] = item["chosen"]
                
        print(f"   -> {len(rule_map)}개의 룰맵 항목 로드 완료 (공백/대소문자 유지)")
        return rule_map

    except Exception as e:
        print(f"❌ 룰맵 로드 실패: {e}")
        return {}

def process_keywords(kywd_str, rule_map):
    """
    키워드 변환: 검색 키 정규화 없이 매칭 -> 최종 결과만 소문자화
    """
    if not kywd_str or not isinstance(kywd_str, str):
        return ""

    # 1. 쉼표로 분리
    keywords = [k.strip() for k in kywd_str.split(',')]
    processed_list = []

    for k in keywords:
        if not k: continue
        
        # [변경점] 검색 키를 만들 때 공백제거/소문자화를 하지 않음
        # 입력된 단어 그대로(앞뒤 공백만 제거) 룰맵에서 찾음
        search_key = k 
        
        # 2. 룰맵 적용
        # 룰맵에 있으면 매핑된 값(standard), 없으면 원본(k) 사용
        final_word = rule_map.get(search_key, k)
        
        # 3. 최종 결과 소문자 변환
        # - 영어: 소문자로 변환됨 (Deep Learning -> deep learning)
        # - 한글: 변환되지 않음 (인공지능 -> 인공지능)
        final_word = final_word.lower()
        
        # 중복 방지
        if final_word not in processed_list:
            processed_list.append(final_word)

    # 4. 다시 문자열로 결합
    return ", ".join(processed_list)

def main():
    print(f"🚀 최종 데이터 키워드 정제 시작 (단순 매칭)")

    # 1. 룰맵 로드
    rule_map = load_rule_map(rule_map_path)

    # 2. 데이터 로드
    if not os.path.exists(input_json_path):
        print(f"❌ 오류: 입력 파일({input_json_path})이 없습니다.")
        return

    with open(input_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 데이터 구조 확인
    if isinstance(data, dict):
        node_list = data.get("NODE_LIST", [])
        final_structure = data
    elif isinstance(data, list):
        node_list = data
        final_structure = {"NODE_LIST": data}
    else:
        print("❌ 데이터 구조를 인식할 수 없습니다.")
        return

    # 3. 키워드 변환 작업
    count = 0
    for node in node_list:
        original_kywd = node.get("KYWD", "")
        
        if original_kywd:
            new_kywd = process_keywords(original_kywd, rule_map)
            
            if new_kywd != original_kywd:
                node["KYWD"] = new_kywd
                count += 1

    # 4. 저장 (indent=4 적용)
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_structure, f, ensure_ascii=False, indent=4)

    print("\n" + "="*50)
    print(f"🎉 작업 완료!")
    print(f"- 총 논문 수: {len(node_list)}건")
    print(f"- 키워드가 변환된 논문: {count}건")
    print(f"- 저장된 파일: {output_json_path}")
    print("="*50)

if __name__ == "__main__":
    main()

🚀 최종 데이터 키워드 정제 시작 (단순 매칭)
📂 룰맵 로딩 중...
   -> 2021개의 룰맵 항목 로드 완료 (공백/대소문자 유지)

🎉 작업 완료!
- 총 논문 수: 32952건
- 키워드가 변환된 논문: 32697건
- 저장된 파일: SSU_Datathon2025_공학분야_62199_Real_Final.json


In [10]:
# KYWD 필드에서 키워드 확인

import json
from collections import Counter
import os

# 1. 분석할 파일 경로
target_json_path = 'SSU_Datathon2025_공학분야_62199_Real_Final.json'

def extract_and_show_keywords():
    print(f"📂 파일 분석 시작: {target_json_path}")
    
    if not os.path.exists(target_json_path):
        print(f"❌ 오류: '{target_json_path}' 파일이 없습니다.")
        return

    try:
        # JSON 로드
        with open(target_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # 데이터 구조 확인
        if isinstance(data, dict):
            node_list = data.get("NODE_LIST", [])
        elif isinstance(data, list):
            node_list = data
        else:
            print("❌ 데이터 구조를 인식할 수 없습니다.")
            return

        # 2. 키워드 추출 및 카운팅
        all_keywords = []
        
        for node in node_list:
            kywd_str = node.get("KYWD", "")
            if kywd_str:
                # 쉼표로 분리 후 앞뒤 공백 제거
                keywords = [k.strip() for k in kywd_str.split(',')]
                # 빈 문자열 제외하고 리스트에 추가
                all_keywords.extend([k for k in keywords if k])

        # 빈도수 계산
        keyword_counts = Counter(all_keywords)

        # 3. 결과 출력
        print("\n" + "="*50)
        print(f"📊 키워드 분석 결과")
        print(f"   - 총 추출된 키워드 (중복 포함): {len(all_keywords):,}개")
        print(f"   - 고유 키워드 종류: {len(keyword_counts):,}개")
        print("="*50)
        print(f"{'순위':<5} {'키워드':<40} {'빈도수':<10}")
        print("-" * 60)
        
        # 상위 30개 출력
        for rank, (kwd, count) in enumerate(keyword_counts.most_common(30), 1):
            print(f"{rank:<5} {kwd:<40} {count:<10}")
            
        print("-" * 60)
        
    except Exception as e:
        print(f"❌ 에러 발생: {e}")

if __name__ == "__main__":
    extract_and_show_keywords()

📂 파일 분석 시작: SSU_Datathon2025_공학분야_62199_Real_Final.json

📊 키워드 분석 결과
   - 총 추출된 키워드 (중복 포함): 202,871개
   - 고유 키워드 종류: 123,409개
순위    키워드                                      빈도수       
------------------------------------------------------------
1     deep learning                            1450      
2     machine learning                         1050      
3     artificial intelligence                  709       
4     convolutional neural network             425       
5     object detection                         327       
6     metaverse                                261       
7     virtual reality                          258       
8     covid 19                                 246       
9     big data                                 215       
10    generative ai                            202       
11    blockchain                               192       
12    autonomous driving                       184       
13    reinforcement learning                   181       


In [11]:
import json
import os

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
# 확인하고 싶은 JSON 파일명을 입력하세요
target_json_path = 'SSU_Datathon2025_공학분야_62199_Real_Final.json'

def count_empty_keywords():
    print(f"📂 파일 분석 중... ({target_json_path})")
    
    if not os.path.exists(target_json_path):
        print(f"❌ 오류: 파일({target_json_path})을 찾을 수 없습니다.")
        return

    try:
        with open(target_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # 데이터 구조 유연하게 처리 (딕셔너리 or 리스트)
        if isinstance(data, dict) and "NODE_LIST" in data:
            node_list = data["NODE_LIST"]
        elif isinstance(data, list):
            node_list = data
        else:
            print("❌ 데이터 구조를 인식할 수 없습니다.")
            return

        # 2. 키워드 공백 개수 세기
        total_count = len(node_list)
        empty_count = 0
        
        print(f"   -> 총 {total_count}건의 논문 검사 시작...")

        for node in node_list:
            # KYWD 필드를 가져오고, 앞뒤 공백 제거
            kywd = str(node.get("KYWD", "")).strip()
            
            # 1) 아예 비어있거나("") 
            # 2) "NaN" 문자열이거나 (혹시 남아있을 경우)
            if not kywd or kywd.upper() == "NAN":
                empty_count += 1

        # 3. 결과 출력
        print("\n" + "="*50)
        print(f"📊 분석 결과")
        print(f"   - 전체 논문 수: {total_count}건")
        print(f"   - 키워드 빈 값: {empty_count}건")
        print(f"   - 키워드 채워짐: {total_count - empty_count}건")
        print(f"   - 공백 비율: {(empty_count / total_count * 100):.2f}%")
        print("="*50)

    except Exception as e:
        print(f"❌ 에러 발생: {e}")

if __name__ == "__main__":
    count_empty_keywords()

📂 파일 분석 중... (SSU_Datathon2025_공학분야_62199_Real_Final.json)
   -> 총 32952건의 논문 검사 시작...

📊 분석 결과
   - 전체 논문 수: 32952건
   - 키워드 빈 값: 104건
   - 키워드 채워짐: 32848건
   - 공백 비율: 0.32%


In [12]:
# 전체 키워드 수 확인

import json
import os

# 분석할 파일 경로
input_file = 'SSU_Datathon2025_공학분야_62199_Real_Final.json'

def count_total_keywords():
    print(f"📂 전체 키워드 수량을 집계합니다... ({input_file})")

    if not os.path.exists(input_file):
        print(f"❌ 오류: '{input_file}' 파일을 찾을 수 없습니다.")
        return

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # 데이터 구조 확인
        if isinstance(data, dict):
            node_list = data.get("NODE_LIST", [])
        elif isinstance(data, list):
            node_list = data
        else:
            print("❌ 데이터 구조를 인식할 수 없습니다.")
            return

        # 집계 변수
        total_keyword_count = 0      # 전체 키워드 누적 개수
        papers_with_keywords = 0     # 키워드가 하나라도 있는 논문 수

        for node in node_list:
            kywd_str = node.get("KYWD", "")
            
            # 키워드 문자열이 비어있지 않은 경우
            if kywd_str and isinstance(kywd_str, str):
                # 쉼표(,)로 쪼개고 빈 값은 제외한 뒤 개수 세기
                keywords = [k.strip() for k in kywd_str.split(',') if k.strip()]
                
                count = len(keywords)
                if count > 0:
                    total_keyword_count += count
                    papers_with_keywords += 1

        # 결과 출력
        print("\n" + "="*50)
        print("🔢 [전체 키워드 수량 집계 결과]")
        print("="*50)
        print(f"✅ 총 논문 수            : {len(node_list):,} 건")
        print(f"✅ 키워드가 있는 논문 수 : {papers_with_keywords:,} 건")
        print("-" * 50)
        print(f"🔥 전체 키워드 총 개수 (누적) : {total_keyword_count:,} 개")
        print("-" * 50)
        
        if papers_with_keywords > 0:
            avg = total_keyword_count / papers_with_keywords
            print(f"   (논문 1편당 평균 {avg:.2f}개의 키워드 보유)")
        print("="*50)

    except Exception as e:
        print(f"❌ 에러 발생: {e}")

if __name__ == "__main__":
    count_total_keywords()

📂 전체 키워드 수량을 집계합니다... (SSU_Datathon2025_공학분야_62199_Real_Final.json)

🔢 [전체 키워드 수량 집계 결과]
✅ 총 논문 수            : 32,952 건
✅ 키워드가 있는 논문 수 : 32,848 건
--------------------------------------------------
🔥 전체 키워드 총 개수 (누적) : 202,871 개
--------------------------------------------------
   (논문 1편당 평균 6.18개의 키워드 보유)
